<a href="https://colab.research.google.com/github/nick0411/analysis-of-single-shot-and-few-shot-learning-of-an-ai-model/blob/main/Singleshot_and_Fewshot_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment Template

Please follow the template below to submit part A of your assignment. Make sure all the outputs are provided within the submission file. Make sure to familarise yourself with the assignment brief before.

# Libraries

In [ ]:
!pip install "transformers<5" datasets evaluate rouge_score pandas torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 31.6 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=d7ea22c6de66f287d33b8cf25577463259a42536cdb7aa32bc7fa79fc5598938
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.8.0
    Uninstalling huggingface_hub-1.8.0:
      Successfully uninstalled huggingface_hub-1.8.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalle

# All Pre-requsites (Hugging Face login, data import etc.)

Note: If you use a HuggingFace model, please do not share your access key, make sure all the outputs are visible, you will be marked down if outputs are not provided)

Provide all functions that you might need for processing the data and using the model (refer to Unit 4 tutorial for some hints)

In [ ]:
import torch
import pandas as pd
from datasets import load_dataset
from transformers import pipeline
import evaluate

I use facebook/bart-large-xsum because this model is good at text summarization.

In [ ]:
model_name = "facebook/bart-large-xsum"

summarizer = pipeline("text2text-generation", model=model_name, tokenizer=model_name, device=0 if torch.cuda.is_available() else -1)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/309 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


Load dataset and subtract small parts of it and view them.

In [ ]:
dataset = load_dataset("EdinburghNLP/xsum")

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/300M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/16.4M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/16.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/204045 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11332 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11334 [00:00<?, ? examples/s]

In [ ]:
train_dataset = dataset['train'].shuffle(seed=44).select(range(1000))
test_dataset = dataset['test'].shuffle(seed=44).select(range(1000))

In [ ]:
train_dataset

Dataset({
    features: ['document', 'summary', 'id'],
    num_rows: 1000
})

In [ ]:
test_dataset

Dataset({
    features: ['document', 'summary', 'id'],
    num_rows: 1000
})

# Zero-shot Strategy

I prepared function for zero shot. the function is simple. Prompt has one line request for AI to do and the context. when it summarize, i use truncation for memory issues. and general summarization in one line doesn't need much words. so i set max token to 30.

In [ ]:
def zero_shot_summarize(text):
  prompt = f"""
  Summarize the following news article in one concise sentence.

  Article:
  {text}

  Summary:
  """

  result = summarizer(
      prompt,
      truncation=True,
      max_new_tokens=30,
      min_length=5
      )

  return result[0]["generated_text"]

This is acutually putting into pratice with the dataset i substracted. and the results are stored.

In [ ]:
zero_shot_preds = []

for doc in test_dataset["document"]:
  summary = zero_shot_summarize(doc)
  zero_shot_preds.append(summary)

# Few-shot Strategy

Preparing prompt with 2 examples. and i only use first 200 characters because model can't handle too much token.

In [ ]:
few_shot_examples = ""

for i in range(2):
  few_shot_examples += f"""
  Article: {train_dataset['document'][i][:200]}
  Summary: {train_dataset['summary'][i]}
  """

In [ ]:
def few_shot_summarize(text):
  prompt = f"""
  Write a one sentence summary of the article.

  Examples:
  {few_shot_examples}

  Article: {text[:200]}
  Summary:
  """

  result = summarizer(
      prompt,
      truncation=True,
      max_new_tokens=30,
      min_length=5
      )

  return result[0]["generated_text"]

Apply few shot function to dataset and record the results

In [ ]:
few_shot_preds = []

for doc in test_dataset["document"]:
  summary = few_shot_summarize(doc)
  few_shot_preds.append(summary)

# Prompt Engineering

## Zero-shot

The difference between this and previous is the prompt are more refined and precise. this is to narrow down the scope for AI model and get better results.

In [ ]:
def engineered_zero_shot(text):
  prompt = f"""
  You are a professional journalist.

  Summarize the article below into ONE factual sentence.

  Rules:
  - Maximum 25 words
  - Capture the main event
  - Avoid unnecessary details

  Article:
  {text}

  Summary:
  """

  result = summarizer(prompt, truncation=True, max_new_tokens=30, min_length=5)

  return result[0]["generated_text"]


Similarly, apply to dataset and get results

In [ ]:
engineered_zero_preds = []

for doc in test_dataset["document"]:
  summary = engineered_zero_shot(doc)
  engineered_zero_preds.append(summary)

## Few-shot

Same thing, few shot prompt become more refined and precise

In [ ]:
engineered_examples = ""

for i in range(2):
  engineered_examples += f"""
  Article: {train_dataset['document'][i][:200]}
  One sentence summary: {train_dataset['summary'][i]}
  """

In [ ]:
def engineered_few_shot(text):

  prompt = f"""
  You are a professional news editor.

  Write a concise one-sentence summary under 25 words.

  Examples:
  {engineered_examples}

  Article:
  {text[:200]}

  Summary:
  """

  result = summarizer(
      prompt,
      truncation=True,
      max_new_tokens=30,
      min_length=5
      )

  return result[0]["generated_text"]

Apply it to model and store results

In [ ]:
engineered_few_preds = []

for doc in test_dataset["document"]:
  summary = engineered_few_shot(doc)
  engineered_few_preds.append(summary)

# Evaluation

load rouge matrics and compute the four results with rouge matrics

In [ ]:
rouge = evaluate.load("rouge")

In [ ]:
references = test_dataset["summary"]

In [ ]:
zero_scores = rouge.compute(
    predictions=zero_shot_preds,
    references=references
    )

few_scores = rouge.compute(
    predictions=few_shot_preds,
    references=references
    )

eng_zero_scores = rouge.compute(
    predictions=engineered_zero_preds,
    references=references
    )

eng_few_scores = rouge.compute(
    predictions=engineered_few_preds,
    references=references
    )

Turn the results got after computing with rouge matics into Table data format

In [ ]:
results = pd.DataFrame({
    "Method": [
        "Zero-shot",
        "Few-shot",
        "Prompt Engineered Zero-shot",
        "Prompt Engineered Few-shot"
    ],
    "ROUGE-1": [
        zero_scores["rouge1"],
        few_scores["rouge1"],
        eng_zero_scores["rouge1"],
        eng_few_scores["rouge1"]
    ],
    "ROUGE-2": [
        zero_scores["rouge2"],
        few_scores["rouge2"],
        eng_zero_scores["rouge2"],
        eng_few_scores["rouge2"]
    ],
    "ROUGE-L": [
        zero_scores["rougeL"],
        few_scores["rougeL"],
        eng_zero_scores["rougeL"],
        eng_few_scores["rougeL"]
    ]
})

Here is the results.

In [ ]:
results

,Method,ROUGE-1,ROUGE-2,ROUGE-L
0,Zero-shot,0.424938,0.201567,0.347719
1,Few-shot,0.145990,0.010466,0.119599
2,Prompt Engineered Zero-shot,0.414489,0.194270,0.338848
3,Prompt Engineered Few-shot,0.136478,0.012930,0.108861


# Results Interpretation (max 200 words)

Note: If you reference something from the model outputs, make sure it can be traced (e.g., if you noticed a interesting pattern in the results, make sure to provide an example of that in the outputs).

**Findings**

The findings demonstrate that the zero-shot method had the highest ROUGE scores and the prompt engineered zero-shot was ranked close behind. To the surprise, the few-shot approaches had much lower performance than the zero-shot approaches. This shows that the extra examples were not able to assist the model to improve its summarization performance depending on the model.

---

**Challenges**

The few-shot methods got significantly lower ROUGE scores. This could possibly be because of the long prompt with several examples that confuses the model. Because the facebook/bart-large-xsum model is generally pre-trained to summarize and not to be prompted in the instruction manner, the inclusion of few-shot examples can decrease the ability of the model to concentrate on the target article. In other instances, the produced summaries might also be having duplicated patterns with that of the sample summaries instead of summarizing the new document in an efficient manner.

---
**Benchmark vs Actual Performance**

In general, it is assumed that few-shot prompting will lead to better results as it will give examples of the desired output format. This time, however, the results were not in line with that. It is probably because the BART model is better adapted to direct summarization challenges, as opposed to instruction-following prompts, so zero-shot summarization is more efficient.